# Logistische Regression

Das ist das erste Notebook, das sich mit Klassifikationsmodelle trainiert, also statt einen kontinuierlichen Wert vorherzusagen, ordnen wir Datenpunkte einer Klasse zu.

Wir fokusieren auf auf die Modellfamilie *logistische Regression*.

<img alt="Palmer Penguins" height="50%" src="https://allisonhorst.github.io/palmerpenguins/reference/figures/lter_penguins.png" width="50%">

Wir werden einen Teil des Datensatzes [Palmer Penguins](https://archive.ics.uci.edu/dataset/690/palmer+penguins-3) anwenden. Dieser Datensatz ist eine kuratierte Sammlung von Größenmessungen, Beobachtungen zu Gelegen und Blutisotopenverhältnissen von 344 ausgewachsenen Adélie-, Zügel- und Eselspinguinen, die auf Inseln des Palmer-Archipels in der Antarktis beobachtet wurden. Die Daten wurden ursprünglich zwischen 2007 und 2009 von Dr. Kristen Gorman und im Rahmen des LTER-Programms der Palmer-Station erhoben.

<font color="grey" size="2">Kunstwerk von @allison_horst</font>

## Benötigte Module

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## Hilfsfunktionen

In [ ]:
def normalize_features(x_train: np.ndarray, x_val: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Z-Score-Normalisierung.
    
    :param x_train: Trainingssatz mit normierten Merkmale als Numpy-Array der Form `(n, p)`, wobei
                    `n` die Anzahl an Beobachtungen ist und `p` - die Anzahl an Merkmalen. Spalte
                    `0` muss die Bias-Spalte sein.
    :param x_val: Validierungssatz mit normierten Merkmale als Numpy-Array der Form `(m, p)`, wobei
                  `m` die Anzahl an Beobachtungen ist und `p` - die Anzahl an Merkmalen. Spalte `0`
                  muss die Bias-Spalte sein.
    :return: Tupel `(x_train_norm, x_val_norm, mu, sigma)`.
    """
    mu = x_train.mean(axis=0)
    sigma = x_train.std(axis=0)
    sigma[sigma == 0] = 1  # Division durch Null vermeiden
    x_train_norm = (x_train - mu) / sigma
    x_val_norm = (x_val - mu) / sigma
    return x_train_norm, x_val_norm, mu, sigma


def train_test_split(x: np.ndarray, y: np.ndarray, train_ratio: float) -> \
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Teilt einen Datensatz in einen Trainings- und einen Testsatz auf.

    :param x: Eingabewerte als Numpy-Array der Form `(n, p)`, wobei
              `n` die Anzahl an Beobachtungen ist und `p` - die Anzahl an Merkmalen.
    :param y: Ausgabewerte für die Beobachtungen im Trainingssatz.
    :param train_ratio: Anteil der Beobachtungen zur Bildung des Trainingssatzes.
    :return: Tupel `(x_train, y_train, x_test, y_test)`.
    """
    # Elemente mischen
    generator = np.random.default_rng(19751979)
    indices = generator.permutation(x.shape[0])

    # Trainings- und Testsatz erzeugen
    train_size = int(train_ratio * indices.size)
    indices_train = indices[:train_size]
    indices_test = indices[train_size:]
    x_train, y_train = x[indices_train, :], y[indices_train]
    x_test, y_test = x[indices_test, :], y[indices_test]
    return x_train, y_train, x_test, y_test


def add_bias(x: np.ndarray) -> np.ndarray:
    """
    Fügt Bias-Spalte zur Merkmal-Matrix hinzu.

    :param x: Merkmalsmatrix mit `n` Beobachtungen als Array der Form `(n, p)`.
    :return: Array der Form `(n, p + 1)` in dem die erste Spalte nur Eines beinhaltet.
    """
    bias = np.ones((x.shape[0], 1), dtype=x.dtype)
    return np.hstack((bias, x))

## Teil 1: Daten laden

Die Datei `penguins.csv` beinhaltet Messungen von 3 Arten von Pinguinen: *Adelie*, *Chinstrap*, *Gentoo*. Für jeden Pinguin werden 4 Merkmale gemessen - Schnabellänge, Schnabeltiefe, Flossenlänge und Körpermasse.

Die ersten 4 Zeilen der Datei sind:

`species,bill length,bill depth,flipper length,body mass`<br />
`Chinstrap,45.4,18.7,188,3525`<br />
`Gentoo,42.7,13.7,208,3950`<br />
`Adelie,38.1,17.6,187,3425`

In [ ]:
def load_penguins(file_csv: str) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Lädt eine vereinfachte Version des "Palmer Penguins"-Datensatzes.
    :param file_csv: Pfad zur CSV-Datei mit den folgenden 5 Spalten:
                     species, bill length, bill depth, flipper length, body mass.
    :return: Tupel `(x, y, species)`, wobei:
             `x` einen Numpy-Array der Form `(n, 4)` ist, der die Merkmalswerte beinhaltet;
             `y` einen Numpy-Array der Form `(n,)` ist, der die Kategorieindizes beinhaltet;
             `species` eine Liste mit allen Pinguinarten ist.
    """
    expected_header = ['species', 'bill length', 'bill depth', 'flipper length', 'body mass']
    delimiter = ','

    # Alle Zeilen der Datei parsen
    x = []  # Merkmalswerte
    penguins = []  # alle Pinguinarten
    with open(file_csv, 'r') as file_handle:
        for index_line, line in enumerate(file_handle):
            line = line.strip()
            if index_line == 0:
                if line != delimiter.join(expected_header):
                    raise ValueError('Unerwartete Header-Zeile')
            elif line != '':
                values = line.split(delimiter)
                if len(values) != len(expected_header):
                    raise ValueError(f'Unerwartete Spalenzahl in Zeile {index_line + 1}')
                x.append([float(x) for x in values[1:]])
                penguins.append(values[0])

    # Artennamen in Indizes umwandeln
    species = list(set(penguins))
    species.sort()
    y = [species.index(current_name) for current_name in penguins]

    # Listen mit Zahlen zu Numpy-Arrays konvertieren
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.uint16)
    return x, y, species


# Bitte Pfad zur CSV-Datei anpassen
file_dataset = 'penguins.csv'
x_penguins, y_penguins, species = load_penguins(file_dataset)
feature_names = [
    'Schnabellänge (mm)', 'Schnabeltiefe (mm)',
    'Flossenlänge (mm)', 'Körpermasse (g)']

# Info über den Datensatz:
print(f'Dimensionalität von x: {x_penguins.shape}')
print(f'Dimensionalität von y: {y_penguins.shape}')
print(f'          Pinguinarte: {species}')

### Ubung

Zeigen Sie die ersten 3 Reihen von `x` sowie die ersten 3 Ausgabewerte (Klassenindizes) in `y`.

In [ ]:
print('Die ersten 3 Beobachtungen:')
print(x_penguins[:3, :])
print(f'Die ersten 3 Ausgabewerte: {y_penguins[:3]}')
print(f'Anzahl Beobachtungen pro Klasse: {np.bincount(y_penguins)}')

## Teil 2: Binäre Klassifikation

Logistische Regression ist ursprünglich ein binärer Klassifikator.
    
Wir fangen mit zwei Klassen (Adelie und Chinstrap) und zwei Features (Schnabellänge und Schnabeltiefe) an, damit wir die Entscheidungsgrenze in 2D plotten können.

In [ ]:
binary_mask = y_penguins <= 1  # Adelie = 0, Chinstrap = 1
x_binary = x_penguins[binary_mask, :2]  # die ersten 2 Features
y_binary = y_penguins[binary_mask]
print(f'{y_binary.size} Beobachtungen wurden ausgewählt')

### Visualisierung

Mithilfe eines Streudiagramms (Scatterplot) können wir überprüfen, ob die Klassen durch eine Linie trennbar sind.

In [ ]:
_, ax = plt.subplots(figsize=(4.5, 4), dpi=100)
for cls, name, marker in [(0, species[0], 'o'), (1, species[1], '^')]:
    mask = y_binary == cls
    ax.scatter(x_binary[mask, 0], x_binary[mask, 1],
               label=name, marker=marker, edgecolors='k', linewidths=0.5, s=30)
ax.set(axisbelow=True, xlabel=feature_names[0], ylabel=feature_names[1])
ax.legend()
ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

### Train-Validation-Split und Normalisierung

Genau wie bei linearen Regression, hilft es sehr, die Merkmalswerte vor dem Training zu normalisieren.

In [ ]:
x_binary_train, y_binary_train, x_binary_val, y_binary_val =\
    train_test_split(x_binary, y_binary, train_ratio=0.8)

x_binary_train_n, x_binary_val_n, mu_binary, sigma_binary =\
    normalize_features(x_binary_train, x_binary_val)

### Sigmoid

Die Sigmoid-Funktion (auch: logistische Funktion) bildet jeden reellen Wert $z$ auf das Intervall $(0, 1)$ ab:

$\sigma(z) = \frac{1}{1 + e^{-z}}$

Die Interpretation ist $\sigma(z) = P(y=1 | x)$, also die Wahrscheinlichkeit, dass der Datenpunkt zur Klasse $1$ gehört.

Einpaar wichtige Eigenschaften sind:
- $\sigma(0) = 0.5$
- $\sigma(z) + \sigma(-z) = 1$
- $\sigma(z) → 1 \; \text{für} \; z → ∞$
- $\sigma(z) → 0 \; \text{für} \; z → -∞$

### Übung

Implementieren Sie die Sigmoid-Funktion. Verwenden Sie [`np.clip`](https://numpy.org/doc/stable/reference/generated/numpy.clip.html) um die Inputwerde im Interval \[-500, +500\] zu halten und damit die Ausgabewerte numerisch stabil zur halten. 

In [ ]:
def sigmoid(z):
    """
    Berechnet die Sigmoid-Funktion.
    
    :param z: Realer Wert oder Numpy-Array von realen Werten.
    :return: σ(z)
    """
    z_clipped = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z_clipped))

# Schnelltest
epsilon = 1e-10
print(f'        σ(0) = 0.5: {np.abs(sigmoid(0) - 0.5) < epsilon}')
print(f'σ(7) + σ(-7) = 1.0: {np.abs(sigmoid(7) + sigmoid(-7) - 1) < epsilon}')

### Hausaufgabe

Erstellen Sie ein Diagram der Sigmoid-Funktion $\sigma(z)$ für $z \in [-6, 6]$.

In [ ]:
pass

### Log-Odds (Logit)

Die Umkehrfunktion der Sigmoid heißt *Logit* oder *log-odds*:

$logit(p) = log(p / (1 - p))$

Wenn wir also $z = w^T x$ berechnen, berechnen wir eigentlich die Log-Odds: wie viel wahrscheinlicher ist Klasse $1$ als Klasse $0$ auf einer logarithmischen Skala?

| $z$ |  Odds  | Interpretation                         |
|:---:|:------:|:---------------------------------------|
|  0  |   1:1  | 50/50                                  |
|  1  | ~2.7:1 | Klasse 1 ist ~2.7 mal wahrscheinlicher |
|  2  | ~7.4:1 | Klasse 1 ist ~7.4 mal wahrscheinlicher |
| -2  | ~1:7.4 | Klasse 0 ist ~7.4 mal wahrscheinlicher |

Das untenstehende Code-Snippet veranschaulicht den Zusammenhang zwischen Log-Odds, Odds Ratio und Wahrscheinlichkeit.

In [ ]:
for z in range(-3, 4):
    p = sigmoid(z)
    odds = p / (1 - p)
    print(f'z = {z:+d} -> Odds = {odds:.2f}:1 -> P(y=1) = {p:.4f}')
del z, p, odds

### Verlustfunktion und Gradient

Für logistische Regression maximieren wir die Log-Likelihood — oder äquivalent: wir minimieren die binäre Kreuzentropie:

$L(\mathbf{w}) = -\frac{1}{n} \sum_{i}{[y_i \cdot log(\hat{y}_i) + (1 - y_i) \cdot log(1 - \hat{y}_i)]}$

wobei $\hat{y}_i = \sigma(\mathbf{w}^T \mathbf{x}_i)$ die vorhergesagte Wahrscheinlichkeit ist.

Der Gradient lautet:

$\triangledown L(\mathbf{w}) = \frac{1}{n} \mathbf{X}^T (\hat{\mathbf{y}} - \mathbf{y})$

Das sieht genau so aus wie bei linearer Regression, nur dass $\hat{y}$ jetzt durch die Sigmoid-Funktion geht.

### Übung

Implementieren Sie die Berechnung des Verlustwertes (binäre Kreuzentropie) und des Gradienten.

Hinweis: Verwenden Sie `np.clip` auf die vorhergesagten Wahrscheinlichkeiten, weil $log(0)$ undefiniert ist. Als $\varepsilon$ passt der Wert `1e-12`.

In [ ]:
def compute_loss_and_gradient(x: np.ndarray, y: np.ndarray, w: np.ndarray) -> tuple:
    """
    Berechnet die binäre Kreuzentropie und den Gradienten.

    :param x: Feature-Matrix mit Bias-Spalte als Numpy-Array der Form `(n, d+1)`.
    :param y: binäre Labels als Numpy-Array der Form `(n,)`.
    :param w: Gewichtsvektor als Numpy-Array der Form `(d+1,)`.
    :return: binäre Kreuzentropie und Gradient als Tupel `(loss, gradient)`.
    """
    epsilon = 1e-12
    y_hat = sigmoid(x @ w)
    
    # Numerisch stabil: Clipping gegen log(0)
    y_hat_safe = np.clip(y_hat, epsilon, 1 - epsilon)

    loss = -np.mean(y * np.log(y_hat_safe) + (1 - y) * np.log(1 - y_hat_safe))
    gradient = (1 / x.shape[0]) * x.T @ (y_hat - y)
    return loss, gradient

### Gradientenabstieg

Jetzt setzen wir alles zusammen: Wir trainieren ein logistisches Regressionsmodell mit Gradientenabstieg.

In [ ]:
def train_logistic_regression(x: np.ndarray, y: np.ndarray, learning_rate: float, epoch_count: int) \
    -> tuple[np.ndarray, np.ndarray]:
    """
    Trainiert binäre logistische Regression mit Gradientenabstieg.

    :param x: normalisierte Markmalswerte als Array der Form `(n, p)`. Spalte
              `0` muss die Bias-Spalte sein.
    :param y: Label-Vektor als Array der Form `(n,)`.
    :param learning_rate: Lernrate (Schrittgrösse).
    :param epoch_count: Anzahl der Iterationen.
    :return: Modellparameter und Verlustwerte nach jeder Epoche als Tupel
             `(w, loss_history)`.
    """
    w = np.zeros(x.shape[1])  # Startgewichte: alles Null
    loss_history = np.empty(epoch_count, dtype=np.float64)

    for index_epoch in range(epoch_count):
        loss, gradient = compute_loss_and_gradient(x, y, w)
        w -= learning_rate * gradient
        loss_history[index_epoch] = loss

    return w, loss_history


# Training starten
x_binary_train_b = add_bias(x_binary_train_n)
x_binary_val_b = add_bias(x_binary_val_n)
w_binary, losses = train_logistic_regression(
    x_binary_train_b, y_binary_train, learning_rate=1.0, epoch_count=150)

### Lernkurve visualisieren

Wenn wir die Verlustwerte über alle Epochen hinweg visualisieren, können wir erkennen, ob der Algorithmus gegen ein Minimum konvergiert ist.

In [ ]:
def plot_losses(losses: np.ndarray, metric: str, yscale: str = 'linear'):
    """
    Visualisiert die Lernkurve eines Modells.

    :param losses: Werte der Verlustfunktion nach jeder Epoche.
    :param metric: Verlustmetrik; diese wird als Beschriftung der x-Achse verwendet.
    :param yscale: Skala der y-Achse.
    """
    _, ax = plt.subplots(figsize=(8, 3), dpi=100)
    xs = np.arange(losses.size) + 1
    ax.plot(xs, losses, '-', linewidth=2)
    ax.set(axisbelow=True, xlabel='Epoche', xlim=[0, xs[-1]], ylabel=metric, yscale=yscale)
    plt.grid(color='#A0A0A0', linewidth=0.5, linestyle='--')
    plt.tight_layout()
    plt.show()


plot_losses(losses, metric='Kreuzentropie', yscale='log')

### Genauigkeit

Wir können das Modell auf den Validierungssatz evaluieren.

In [ ]:
y_binary_pred_prob = sigmoid(x_binary_val_b @ w_binary)
y_binary_pred = (y_binary_pred_prob >= 0.5).astype(np.uint16)
accuracy_binary = np.mean(y_binary_pred == y_binary_val)
print(f'                      Genauigkeit: {accuracy_binary * 100:.1f}%')
print(f'Beobachtungen im Validierungssatz: {y_binary_val.size}')

### Entscheidungsgrenze visualisieren

Die Entscheidungsgrenze ist dort, wo $P(y=1 | \mathbf{x}) = 0.5$, also wo $\sigma(\mathbf{w}^T \mathbf{x}) = 0.5$, also wo

$\mathbf{w}^T \mathbf{x} = 0$.

Im zweidimensionalen Raum ist das eine Gerade:

$w_0 + w_1 x_1 + w_2 x_2 = 0$

$\Rightarrow \; x_2 = -\frac{w_0 + w_1 x_1}{w_2}$


In [ ]:
def plot_decision_boundary_2d(X: np.ndarray, y: np.ndarray,
                              mu: np.ndarray, sigma: np.ndarray,
                              class_names: list[str],
                              feature_names: list[str],
                              w: np.ndarray):
    """
    Plottet die Datenpunkte und die Entscheidungsgrenze in 2D.

    :param x: Originaldaten (nicht normalisiert) als Array der Form `(n, p)`.
    :param y: Labelindizes der Beobachtungen als Array der Form `(n,)`.
    :param mu: Durchschnittswerte der Merkmale als Array der Form `(p,)`.
    :param sigma: Standardabweichungen der Merkmale als Array der Form `(p,)`.
    :param class_names: Liste der Klassennamen.
    :param feature_names: Liste der Merkmalnamen.
    :param w: Gewichtsvektor der logistischen Regression, trainiert auf
              normalisierten Daten.
    """
    _, ax = plt.subplots(figsize=(4.5, 4.5), dpi=100)

    # Datenpunkte plotten
    markers = ['o', '^', 's']
    colors = plt.cm.Set2.colors
    for cls in range(len(class_names)):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], label=class_names[cls],
                    marker=markers[cls], edgecolors='k', linewidths=0.5,
                    s=30, color=colors[cls])

    # Entscheidungsgrenze: w0 + w1*z1 + w2*z2 = 0
    # => w0 + w1*(x1-mu1)/s1 + w2*(x2-mu2)/s2 = 0
    # => x2 = -(w0 + w1*(x1-mu1)/s1) * s2/w2 + mu2
    x1_range = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200)
    if abs(w[2]) > 1e-10:
        x2_boundary = -(w[0] + w[1] * (x1_range - mu[0]) / sigma[0]) \
                       * sigma[1] / w[2] + mu[1]
        ax.plot(x1_range, x2_boundary, 'k--', linewidth=2, label='Entscheidungsgrenze')

    ax.set(axisbelow=True, xlabel=feature_names[0], ylabel=feature_names[1])
    ax.legend()
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    plt.show()


plot_decision_boundary_2d(
    x_binary, y_binary, mu_binary, sigma_binary,
    species[:2], feature_names[:2], w_binary)

## Teil 3: Mehrklassen-Klassifikation

Wir verwenden die Strategie One-vs-rest (OvR) für den Übergang von 2 Klassen auf 3 Klassen (oder mehr).

Für die drei Klassen trainieren wir 3 binäre Klassifikatoren:
1. "Ist es Adelie?"  (Adelie=1, Rest=0)
2. "Ist es Chinstrap?" (Chinstrap=1, Rest=0)
3. "Ist es Gentoo?"  (Gentoo=1, Rest=0)

Für eine neue Beobachtung lassen wir *alle* 3 Klassifikatoren laufen und wählen die Klasse mit der höchsten Wahrscheinlichkeit.

Die folgende Funktion implementiert den Trainingsprozess bei mehrklassen-Klassifikation.

In [ ]:
def train_one_vs_rest(x: np.ndarray, y: np.ndarray, learning_rate: float, epoch_count: int):
    """
    Trainiert logistische Regression mit Gradientenabstieg.

    :param x: normalisierte Markmalswerte als Array der Form `(n, p)`. Spalte
              `0` muss die Bias-Spalte sein.
    :param y: Labelindizes der Beobachtungen als Array der Form `(n,)`.
    :param learning_rate: Lernrate (Schrittgrösse).
    :param epoch_count: Anzahl der Iterationen.
    :return: Modellparameter und Verlustwerte nach jeder Epoche als Tupel
             `(w, accuracies)`.
    """
    class_count = y.max().item() + 1

    # Ausgabewerte für alle Klassifikatoren erstellen
    y_binary = np.empty((y.size, class_count), dtype=y.dtype)
    for index_class in range(class_count):
        y_binary[:, index_class] = y == index_class

    # Gewichtmatrix und Array für Genauigkeitswerte initialisieren
    w = np.zeros((x.shape[1], class_count), dtype=np.float64)
    accuracies = np.empty(epoch_count, dtype=np.float64)

    for index_epoch in range(epoch_count):
        # Einen Schritt für alle Klassifikatoren machen
        for index_class in range(class_count):
            _, gradient =\
                compute_loss_and_gradient(x, y_binary[:, index_class], w[:, index_class])
            w[:, index_class] -= learning_rate * gradient
            
        # Trainingsgenauigkeit berechnen
        y_hat = sigmoid(x @ w).argmax(axis=1)
        accuracies[index_epoch] = np.mean(y == y_hat)

    return w, accuracies

### Trainieren und Ergebnisse visualisien

In [ ]:
# Train/Test-Split auf dem gesamten Datensatz
x_train, y_train, x_val, y_val =\
    train_test_split(x_penguins, y_penguins, train_ratio=0.8)

# Normalisieren
x_train_n, x_val_n, mu_mc, sigma_mc = normalize_features(x_train, x_val)

# Bias hinzufügen
x_train_b = add_bias(x_train_n)
x_val_b  = add_bias(x_val_n)

# Trainieren
weights, accuracies =\
    train_one_vs_rest(x_train_b, y_train, learning_rate=1.0, epoch_count=100)

# Genauigkeitswerte visualisieren
plot_losses(accuracies, 'Genauigkeit')

## Teil 4: Evaluierungsmetriken

### Konfusionsmatrix

Die Konfusionsmatrix zeigt, welche Klassen verwechselt werden. Zeilen = wahre Klasse, Spalten = vorhergesagte Klasse. Auf der Diagonale stehen die korrekt klassifizierten Beobachtungen.

### Übung

Implementieren Sie die Funktion zur Berechnung der Konfusionsmatrix.

`cm[i, j] = `Anzahl der Datenpunkte, deren wahre Klasse `i` ist und die als Klasse `j` vorhergesagt wurden.

In [ ]:
def compute_confusion_matrix(y: np.ndarray, y_hat: np.ndarray) -> np.ndarray:
    """
    Berechnet die Konfusionsmatrix.

    :param y: wahre Labels als Array der Form `(n,)`.
    :param y_hat: vorhergesagte Labels als Array der Form `(n,)`.
    :return: Konfusionsmatrix als Integer-Array der Form `(C, C)`.
    """
    class_count = max(y.max().item(), y_hat.max().item()) + 1
    result = np.zeros((class_count, class_count), dtype=np.uint32)
    for class_index_true, class_index_predicted in zip(y, y_hat):
        result[class_index_true, class_index_predicted] += 1
    return result


# Konfusionsmatrix für den Validierungssatz berechnen
outputs_val = sigmoid(x_val_b @ weights)
y_hat_val = outputs_val.argmax(axis=1)
cm = compute_confusion_matrix(y_val, y_hat_val)
print(cm)

# Die Ausgabewerte für das erste falschklassifizerte Beispiel schauen
i = np.where(y_val != y_hat_val)[0][0].item()
print(f'\nEchtes Label für Beobachtung {i} = {y_val[i]}')
print(f'     Outputs für Beobachtung {i} = {outputs_val[i, :]}')

### Metriken basierend auf der Konfusionsmatrix

Für jede Klasse $k$:
- TP (True Positives)  = `cm[k, k]`
- FP (False Positives) = Summe der Spalte `k`, minus `cm[k, k]`
- FN (False Negatives) = Summe der Zeile `k`, minus `cm[k, k]`
- TN (True Negatives)  = alles andere


**Precision** = TP / (TP + FP)  — "Wie viele der Vorhersagen waren richtig?"

**Recall**    = TP / (TP + FN)  — "Wie viele der echten Fälle wurden gefunden?"
             (= Sensitivität = True Positive Rate)

**F1-Score**  = 2 · (Precision · Recall) / (Precision + Recall)

**Spezifität** = TN / (TN + FP) — "Wie gut erkennt das Modell Nicht-Klasse-k?"


In [ ]:
def compute_metrics_per_class(cm: np.ndarray, class_names: list[str]):
    """
    Berechnet Precision, Recall, F1, Sensitivität und Spezifität pro Klasse
    aus der Konfusionsmatrix.

    :param cm: Konsfusionsmatrix als Array der Form `(C, C)`.
    :param class_names: Klassennamen als Liste der Länge `C`.
    :return: ...
    """
    metrics = {}

    for k, class_name in enumerate(class_names):
        tp = cm[k, k]
        fp = cm[:, k].sum() - tp
        fn = cm[k, :].sum() - tp
        tn = cm.sum() - tp - fp - fn

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = 2 * precision * recall / (precision + recall) \
                    if (precision + recall) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        metrics[class_name] = {
            'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            'Precision': precision,
            'Recall (Sensitivität)': recall,
            'Spezifität': specificity,
            'F1-Score': f1
        }
    return metrics


# Metriken berechnen und auf der Konsole ausgeben
all_metrics = compute_metrics_per_class(cm, species)
for class_name, metrics in all_metrics.items():
    print(class_name)
    for metric_name, metric_value in metrics.items():
        print(f'{metric_name}: {metric_value}')
    print('')

## Hausaufgabe

Bauen Sie das beste Modell, um die Pinguinarten auf dem **Testdatensatz** vorherzusagen.

Die Testdatei enthält nur die Features — keine Artenlabels. Sie müssen die Wahrscheinlichkeiten für jede Art vorhersagen
und als Text hochladen.

Format der Ausgabe:

`0.92,0.05,0.03`<br/>
`0.01,0.02,0.97`

Die Spalten bezeichnen Adelie, Chinstrap und Gentoo (in dieser Rheinfolge).

Sobald Sie einen Klassifikator trainiert und ihn für die Beobachtungen im Testsatz ausgeführt haben, können Sie das Ergebnis auf unserer [Website zum Klassifikatorwettbewerb](http://ai-epi.com/competition/submit.php) einreichen.

In [ ]:
# Schritt 1: Testdaten laden, bitte Pfad anpassen
x_test = np.loadtxt('penguins-test.csv', delimiter=',', skiprows=1)
print(x_test)

# Schritt 2: Logistische Regression trainieren und auf dem Testsatz anwenden

# Schritt 3: Die Vorhersagen als Wahrscheinlichkeitswerte ausgeben

# Schritt 4: Hochladen
